In [5]:
import tkinter as tk
from tkinter import filedialog, ttk
import time

def browse_file():
    file_path = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx;*.xls")])
    entry_file.delete(0, tk.END)
    entry_file.insert(0, file_path)

def browse_save_file():
    file_path_Save = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel files", "*.xlsx;*.xls")])
    entry_save_file.delete(0, tk.END)
    entry_save_file.insert(0, file_path_Save)

def calculate():
    file_path = entry_file.get()
    file_path_Save = entry_save_file.get()
    import pandas as pd
    
    # โหลดไฟล์ Excel
    xxx = pd.read_excel(file_path)
    
    # ลบคอลัมน์ที่ไม่ต้องการ
    xxx = xxx.drop(columns=['จำนวนคงเหลือ'])
    
    # เพิ่มคอลัมน์ใหม่
    xxx["จำนวนคงเหลือ"] = 0
    xxx["Order Lines/Product/Barcode2"] = xxx['Order Lines/Product/Barcode']
    xxx['Order Lines/Product/Barcode2'] = xxx['Order Lines/Product/Barcode2'].str.replace(' ผลรวม', '', regex=True)
    
    # ดึงค่าใบเสร็จที่ไม่ซ้ำ
    unique_receipts = xxx['Order Lines/Product/Barcode2'].unique()
    
    # วนลูปทีละใบเสร็จ
    for receipt in unique_receipts:
        receipt_data = xxx[xxx["Order Lines/Product/Barcode2"] == receipt].copy()
    
        if receipt_data.empty:
            continue  # ข้ามถ้าไม่มีข้อมูล
    
        # กำหนดค่าเริ่มต้นให้ remaining เป็นค่า 'Qty. OH 31.12.24' ของแถวแรก
        first_index = receipt_data.index[0]
        remaining = receipt_data.at[first_index, 'Stock On Hand']
        first_qty = receipt_data.at[first_index, 'Stock On Hand']
        
    
        # วนลูปทีละแถว
        for i, index in enumerate(receipt_data.index):  
            AA = receipt_data.at[index, 'Qty.ซื้อ']  # ค่าของ AA
            
            if remaining > AA:
                receipt_data.at[index, 'จำนวนคงเหลือ'] = AA  # ใส่ AA ลงใน CC ก่อน
                remaining -= AA  # คำนวณค่าที่เหลือ
            else:
                receipt_data.at[index, 'จำนวนคงเหลือ'] = remaining  # ถ้า remaining <= AA ให้ใส่ค่า remaining ลงใน CC
                remaining = 0  # ค่าหมดแล้ว
    
            # ถ้าหลังจากลบแล้ว remaining เป็น 0 → **จบการทำงานเลย**
            if remaining == 0:
                
                break  # หยุดลูปทันที
    
            # ส่งค่าที่เหลือไปแถวถัดไป (ถ้ายังมีแถวถัดไป)
            if i + 1 < len(receipt_data):
                next_index = receipt_data.index[i + 1]
                receipt_data.at[next_index, 'Stock On Hand'] = remaining  # ใช้ค่าที่เหลือไปเปรียบเทียบกับ AA ในแถวถัดไป
        last_index = receipt_data.index[-1]  # หาค่า index ของแถวสุดท้าย
        receipt_data.at[last_index, 'จำนวนคงเหลือ'] = first_qty
    
        # อัปเดตค่าใน xxx
        xxx.update(receipt_data)
    xxx = xxx.drop(columns=['Order Lines/Product/Barcode2'])
    xxx = xxx[['Order Lines/Product/Barcode', 'Order Lines/Product', 'Receipt Date','Order Reference','Unit Price','Qty.ซื้อ','Stock On Hand','จำนวนคงเหลือ','Diff.']]
    
    xxx["Diff"] = xxx["Stock On Hand"] - xxx["จำนวนคงเหลือ"]
    xxx = xxx.drop(columns=['Diff.'])
    xxx["เช็ค"] = xxx["Diff"] == 0
    xxx["เช็ค"] = xxx["เช็ค"].map({True: "True", False: "False"}) 
    
    xxx.to_excel(file_path_Save)
   
      

# สร้างหน้าต่างหลัก
root = tk.Tk()
root.title("Excel Processor")
root.geometry("400x250")

# ส่วนของการเลือกไฟล์
frame_top = tk.Frame(root)
frame_top.pack(pady=10)

entry_file = tk.Entry(frame_top, width=40)
entry_file.pack(side=tk.LEFT, padx=5)

btn_browse = tk.Button(frame_top, text="Browse", command=browse_file)
btn_browse.pack(side=tk.LEFT)

# ส่วนของการเลือกไฟล์บันทึก
frame_save = tk.Frame(root)
frame_save.pack(pady=10)

entry_save_file = tk.Entry(frame_save, width=40)
entry_save_file.pack(side=tk.LEFT, padx=5)

btn_save_browse = tk.Button(frame_save, text="Browse Save", command=browse_save_file)
btn_save_browse.pack(side=tk.LEFT)

# ปุ่มคำนวณ
btn_calculate = tk.Button(root, text="Cal", command=calculate, width=10)
btn_calculate.pack(pady=10)



# เริ่มต้น GUI
root.mainloop()